In [2]:
import fitz
import os, math
import pandas as pd
import numpy as np
from pathlib import Path
from utils import Helper
utils = Helper()
import fitz

out_dir = r"C:\Users\kaustubh.keny\Projects\OUTPUTS"

In [9]:
json_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\document_Chemfab Alkalis Ltd..json"
json_path = Path(json_path)
root_dir = json_path.parent

In [ ]:
import json
from pathlib import Path
from docling_core.types.doc import DoclingDocument

base_html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Extracted Tables</title>
<style>
body {
    font-family: Arial, sans-serif;
    margin: 30px;
}

.source {
    font-size: 18px;
    font-weight: bold;
    margin-top: 40px;
    margin-bottom: 20px;
}

.figure-title {
    font-size: 16px;
    font-weight: bold;
    margin: 20px 0 10px;
}

table {
    border-collapse: collapse;
    margin-bottom: 40px;
}

td, th {
    border: 1px solid #999;
    padding: 6px;
}

th {
    background-color: #f2f2f2;
}
</style>
</head>
<body>

<h1>Extracted Tables</h1>
"""

json_path = Path(r"C:\Users\kaustubh.keny\Projects\OUTPUTS\output_docling\ops_server")
json_files = list(json_path.rglob("*.json"))
print(f"Found {len(json_files)} JSON files")


for json_file in json_files:
    print(f"Processing: {json_file}")

    with json_file.open("r", encoding="utf-8") as f:
        json_data = json.load(f)["docling_output"]

    doc = DoclingDocument.model_validate(json_data)

    html = base_html

    for idx, table in enumerate(doc.tables, start=1):
        html += f'<div class="figure-title">Table {idx}</div>'
        html += table.export_to_html(doc=doc)
        html += "<br><br>"

    html += """</body></html>"""

    output_file = json_file.with_suffix(".html")

    with output_file.open("w", encoding="utf-8") as f:
        f.write(html)

    print(f"Saved: {output_file}")

# print(f"Saved: {output_file.absolute()}")

In [ ]:
# Column	Sr	Suffix	Prev Cell Suffix	Particular	text_casing	Row Status	Financial	Financial	Financial
# 							Latest Quarter	Previous Quarter	Previous Quarter


In [4]:
file_path = r"Q1 2026 FILES/Quarterly Results 2026 Q1_TYPE.xlsx"
df = pd.read_excel(file_path)

print(f"TOTAL PDFS: {df.shape}")

def max_consecutive_scanned(x):
    max_run = 0
    current_run = 0

    for v in x:
        if v == "scanned":
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0

    return max_run

cons_scan = (
    df.groupby("pdf_name")["type"]
      .apply(max_consecutive_scanned)
      .rename("max_consecutive_scanned")
)

summary = (
    df.groupby("pdf_name")["type"]
      .agg(
          total_pages="count",
          scanned_pages=lambda x: (x == "scanned").sum()
      )
)

summary["scanned_ratio"] = (
    summary["scanned_pages"] / summary["total_pages"]
)

summary["PDF_TYPE_33"] = np.where(
    summary["scanned_ratio"] >= 0.33,
    "SCANNED",
    "TEXT"
)

summary["PDF_TYPE_25"] = np.where(
    summary["scanned_ratio"] >= 0.25,
    "SCANNED",
    "TEXT"
)

summary.join(cons_scan).reset_index().to_excel("RATIO_PDF.xlsx", index=False)

TOTAL PDFS: (34749, 3)


In [5]:
d1 = summary[(summary["PDF_TYPE_25"] == "TEXT") & ( summary["total_pages"] > 1)]
d2 = summary[(summary["PDF_TYPE_25"] == "SCANNED") & ( summary["total_pages"] > 1)]

In [6]:
text_files = d2.index.to_list()
print(len(text_files), text_files[:10])

672 ['3I Infotech Ltd..pdf', 'ACS Technologies Ltd..pdf', 'ADC India Communications Ltd..pdf', 'APT Packaging Ltd..pdf', 'ATV Projects India Ltd..pdf', 'AUDROC Ltd..pdf', 'Aarnav Fashions Ltd..pdf', 'Aarti Drugs Ltd..pdf', 'Aarvi Encon Ltd..pdf', 'Abbott India Ltd..pdf']


In [12]:
folder_path = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
all_df = []
for file in text_files:
    
    fp = os.path.join(folder_path, file)
    
    doc = fitz.open(fp)
    for i in range(1, doc.page_count+1):
        all_df.append({
            "pdf":file,
            "page_n": i
        })

pd.DataFrame(all_df).to_csv("SAMPLE.csv")

In [38]:
import os
import shutil

folder_path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\batch_data"

batch_limit_mb = 100
batch = 1
size_count = 0

# Get files with sizes
files = []

for filename in os.listdir(folder_path):
    fp = os.path.join(folder_path, filename)

    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / (1024 * 1024)
        files.append((filename, size_mb))

# Sort by size descending (largest first)
files.sort(key=lambda x: x[1], reverse=True)

output_folder = os.path.join(out_dir, f"batch_{batch}")
os.makedirs(output_folder, exist_ok=True)

for filename, size_mb in files:

    if size_count + size_mb > batch_limit_mb:
        batch += 1
        size_count = 0

        output_folder = os.path.join(out_dir, f"batch_{batch}")
        os.makedirs(output_folder, exist_ok=True)

    shutil.copy(
        os.path.join(folder_path, filename),
        output_folder
    )

    size_count += size_mb

print(f"Created {batch} batches")

Created 24 batches


In [36]:
import os
import math
import fitz  # PyMuPDF
import pandas as pd

pdf_folder = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
output_folder = os.path.join(out_dir,"batch_data")

os.makedirs(output_folder, exist_ok=True)

# df columns:
# pdf_name
# page_number

df = df.copy()

batch_size = 10
df["batch_pdf_name"] = None

num_batches = math.ceil(len(df) / batch_size)

for batch_no in range(num_batches):

    start_idx = batch_no * batch_size
    end_idx = min(start_idx + batch_size, len(df))

    batch_df = df.iloc[start_idx:end_idx]

    batch_pdf_name = f"batch_{batch_no + 1}.pdf"
    batch_pdf_path = os.path.join(output_folder, batch_pdf_name)

    batch_doc = fitz.open()

    for idx in batch_df.index:

        pdf_name = df.loc[idx, "pdf"]
        page_number = int(df.loc[idx, "page_n"])

        src_pdf_path = os.path.join(pdf_folder, f"{pdf_name}.pdf")

        src_doc = fitz.open(src_pdf_path)

        # convert to zero-based page index
        src_page_idx = page_number - 1

        # copy page
        batch_doc.insert_pdf(
            src_doc,
            from_page=src_page_idx,
            to_page=src_page_idx
        )

        # get newly added page
        page = batch_doc[-1]

        # add label at top-left
        page.insert_text(
            (20, 30),
            f"{pdf_name} | Page {page_number}",
            fontsize=20,
            color=(0, 0, 0)
        )

        src_doc.close()

        # update dataframe
        df.loc[idx, "batch_pdf_name"] = batch_pdf_name

    batch_doc.save(
        batch_pdf_path,
        garbage=4,
        deflate=True,
        clean=True
    )
    batch_doc.close()

print("Batch PDFs created.")

Batch PDFs created.


In [39]:
df.to_csv("batch_info.csv")